In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


In [3]:
df = pd.read_csv('../data/features/features_complete.csv')




features = [
    # --- Existing features ---
    'title_length', 
    'uppercase_words', 
    'sentiment_polarity', 
    'sentiment_subjectivity',
    'category_id',  
    'published_day_of_week_num',
    'hour_of_trending',       # Time of day
    'days_until_trending',    # How long to trend
    'num_emojis',
    'has_emoji',
    'contains_numbers_or_emojis',
    'comments_disabled',
    'ratings_disabled',
    'is_title_english',             # Title language flag

    'top50_pca1',             # PCA embedding of top 50 words
    'top50_pca2',
    'top50_pca3',

    # --- Metadata / flags ---
    'is_published_weekend',   # True if video published on weekend
    'is_trending_weekend',    # True if trending on weekend
]

X = df[features]
y = np.log1p(df['views'])
print(f"Shape: X={X.shape}, y={y.shape}")

Shape: X=(55886, 19), y=(55886,)


In [4]:

X_train, X_test, y_train, y_test  = train_test_split(X,y,test_size=0.2, random_state=42)
# Create RandomForest with reasonable defaults
rf_model = RandomForestRegressor(
    n_estimators=200,      # Number of trees
    max_depth=15,          # Tree depth
    min_samples_split=5,   # Minimum samples to split
    min_samples_leaf=2,    # Minimum samples in leaf
    random_state=42,
    n_jobs=-1,             # Use all cores
    verbose=0
)

print("Training Random Forest...")
# Hint: Fit on X_train, y_train (same as XGBoost)
rf_model.fit(X_train, y_train)

print("✅ Random Forest trained!")


Training Random Forest...
✅ Random Forest trained!


In [5]:
y_pred_rf = rf_model.predict(X_test)

# Calculate metrics
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("\n" + "="*60)
print("RANDOM FOREST PERFORMANCE")
print("="*60)
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE:  {mae_rf:.4f}")
print(f"R²:   {r2_rf:.4f}")


RANDOM FOREST PERFORMANCE
RMSE: 1.3569
MAE:  1.0636
R²:   0.3462


In [6]:
print("\n" + "="*60)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("="*60)

# Get feature importance from RandomForest
rf_importance = rf_model.feature_importances_

# Sort and display
rf_feature_importance = sorted(
    zip(features, rf_importance), 
    key=lambda x: x[1], 
    reverse=True
)

for i, (feature, importance) in enumerate(rf_feature_importance, 1):
    bar = '█' * int(importance * 50)
    print(f"{i:2d}. {feature:<30} {importance:>7.4f}  {bar}")


RANDOM FOREST FEATURE IMPORTANCE
 1. top50_pca2                      0.1694  ████████
 2. title_length                    0.1466  ███████
 3. category_id                     0.1268  ██████
 4. hour_of_trending                0.1023  █████
 5. uppercase_words                 0.1006  █████
 6. sentiment_subjectivity          0.0810  ████
 7. days_until_trending             0.0796  ███
 8. published_day_of_week_num       0.0342  █
 9. num_emojis                      0.0340  █
10. sentiment_polarity              0.0247  █
11. top50_pca1                      0.0221  █
12. has_emoji                       0.0187  
13. top50_pca3                      0.0165  
14. contains_numbers_or_emojis      0.0157  
15. comments_disabled               0.0081  
16. is_trending_weekend             0.0078  
17. ratings_disabled                0.0074  
18. is_published_weekend            0.0044  
19. is_title_english                0.0000  
